<h1>Silver GTFS<h1>

<h2>Imports<h2>

<h3>Spark and Initialization<h3>

In [1]:
import findspark
findspark.init()
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("Warsaw_Bus_Project") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.6.0") \
    .getOrCreate()

<h3>Imports<h3>

In [ ]:
import sys
import gc
from pyspark.sql import DataFrame as SparkDF
sys.path.append('../work')
from config import db_properties,DB_CONFIG,jdbc_url
import os
os.environ["PYARROW_IGNORE_TIMEZONE"] = "1"
import pandas as pd
import numpy as np
import pyspark.pandas as ps
import requests
from pyspark.sql.types import FloatType
from pyspark.sql import Row
api_key = os.getenv("WARSAW_API_KEY")
import psycopg2
from psycopg2 import sql
import pyspark.sql.functions as sf
import zipfile
import urllib.request
from pyspark.testing import assertDataFrameEqual
from pyspark.sql import functions as F

<h3>Creating Silver schema<h3>

In [3]:
conn = psycopg2.connect(dbname="Warsaw_Bus_DB", user="admin", password="admin", host="postgres")
cur = conn.cursor()
cur.execute("CREATE SCHEMA IF NOT EXISTS silver;")
conn.commit()
conn.close()

<h2>Silver Stop_Times Table<h2>

<h3>Extracting Bronze Stop_Times<h3>

In [ ]:
df_stop_times_bronze = spark.read.jdbc(url=jdbc_url, table="bronze.stop_times", properties=db_properties)
df_stop_times_bronze

DataFrame[trip_id: string, arrival_time: string, departure_time: string, stop_id: string, stop_sequence: string, stop_headsign: string, pickup_type: string, drop_off_type: string, shape_dist_traveled: string, timepoint: string]

In [9]:
df_stop_times_bronze = df_stop_times_bronze.drop("stop_headsign")
df_stop_times_bronze = df_stop_times_bronze.where(df_stop_times_bronze.pickup_type != df_stop_times_bronze.drop_off_type)
df_stop_times_bronze.show(5)

+-----------+------------+--------------+-------+-------------+-----------+-------------+-------------------+---------+
|    trip_id|arrival_time|departure_time|stop_id|stop_sequence|pickup_type|drop_off_type|shape_dist_traveled|timepoint|
+-----------+------------+--------------+-------+-------------+-----------+-------------+-------------------+---------+
|9_5_4974089|    24:42:00|      24:42:00|   1931|           48|          1|            0|             31.085|        1|
|9_5_4224384|    04:40:00|      04:40:00|   8748|            0|          0|            1|              0.000|        1|
|5_2_5568543|    21:51:00|      21:51:00|   3398|           19|          1|            0|             10.415|        1|
|5_2_5568544|    22:03:00|      22:03:00|   3398|            0|          0|            1|              0.000|        1|
|5_2_5568544|    22:26:00|      22:26:00|   1930|           18|          1|            0|             10.137|        1|
+-----------+------------+--------------

<h3>Transforming Stop_Times data</h3>